# Evidence-Grounded RAG for AI Agents, Memory, and Evaluation

**Revised code — fresh execution required.** In Colab choose a T4 GPU, then run cells in order. The RAGAS step is slow and saves completed scores so it can resume. The human-review cell waits for your own labels and comments for all 12 answers. No scores or human judgments have been invented or copied from the previous run.

Download the executed notebook after all checks pass; the final cell converts that exact snapshot into matching notebook and HTML files. Review the reference-answer drafts and AI disclosure before submission. This notebook performs no fine-tuning.

## 1. Sources and methodology

| Source | Identifier | Focus |
|---|---|---|
| *AI Agents Need Memory Control Over More Context* | [arXiv:2601.11653](https://arxiv.org/abs/2601.11653) | Memory drift, bounded control, Agent Cognitive Compressor |
| *ProofAgent Harness: Open Infrastructure for Adversarial Evaluation of AI Agents* | [arXiv:2605.24134](https://arxiv.org/abs/2605.24134) | Adversarial trials, multi-juror evaluation, audit |
| *Agentic Systems: A Guide to Transforming Industries with Vertical AI Agents* | [arXiv:2501.00881](https://arxiv.org/abs/2501.00881) | Agent architecture and vertical agents |

Pipeline: **PDF → page-aware extraction → 300-token chunks / 50-token overlap → MiniLM embeddings → ChromaDB → top-4 retrieval → Qwen answer → RAGAS + human checks → PCA**.

We use an open-weight Hugging Face language model rather than a paid API. The model's license and potential hardware constraints should be checked before use.

In [ ]:
# On Colab: Runtime → Change runtime type → T4 GPU (recommended)
%pip -q install "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" "langchain-huggingface>=0.1,<0.2" "langchain-chroma>=0.2,<0.3" "langchain-text-splitters>=0.3,<0.4" "chromadb>=0.6,<0.7" "sentence-transformers>=3.4,<4" "transformers>=4.49,<5" "accelerate>=1.2,<2" "ragas==0.2.15" "datasets>=3,<4" "pymupdf>=1.25" "scikit-learn>=1.4" "matplotlib>=3.8" "pandas>=2.0" "requests>=2.31" "nbconvert>=7"

## 2. Imports, fixed configuration, and runtime checks

In [ ]:
import os, re, gc, json, random, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import fitz
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.decomposition import PCA
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_chroma import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
ROOT = Path('agent_rag_outputs'); ROOT.mkdir(exist_ok=True)
PAPERS = {
  'memory': {'id':'2601.11653','title':'AI Agents Need Memory Control Over More Context'},
  'proofagent': {'id':'2605.24134','title':'ProofAgent Harness: Open Infrastructure for Adversarial Evaluation of AI Agents'},
  'vertical': {'id':'2501.00881','title':'Agentic Systems: A Guide to Transforming Industries with Vertical AI Agents'},
}
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
# GEN_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'  # small open-weight LLM for affordable Colab execution
GEN_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
K=4
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('Embedding:', EMBED_MODEL, '| Generation:', GEN_MODEL)
if not torch.cuda.is_available(): print('CPU mode: generation and RAGAS evaluation may be very slow.')

## 3. Download and extract the *actual full papers*

The URLs are the exact arXiv identifiers from the assignment. PDF pages are retained in metadata for citations. Stop on download failures rather than silently substituting unrelated text.

In [ ]:
def download_pdf(arxiv_id, output):
    if output.exists() and output.read_bytes()[:4] == b'%PDF':
        return output
    urls = [f'https://arxiv.org/pdf/{arxiv_id}', f'https://export.arxiv.org/pdf/{arxiv_id}']
    last_error = None
    for url in urls:
        try:
            res = requests.get(url, timeout=90, headers={'User-Agent':'Mozilla/5.0 (educational RAG study)'})
            res.raise_for_status()
            if not res.content.startswith(b'%PDF'): raise ValueError(f'Non-PDF response from {url}')
            output.write_bytes(res.content)
            return output
        except Exception as e:
            last_error=e
    raise RuntimeError(f'Cannot fetch {arxiv_id}. Upload its arXiv PDF manually as {output} then retry. Last error: {last_error}')

page_docs = []
for key, paper in PAPERS.items():
    path = download_pdf(paper['id'], ROOT / f"{paper['id']}.pdf")
    with fitz.open(path) as pdf:
        for page_idx, page in enumerate(pdf):
            text = page.get_text('text', sort=True)
            text = re.sub(r'[ \t]+', ' ', text)
            text = re.sub(r'\n{3,}', '\n\n', text).strip()
            if len(text) > 80:
                page_docs.append(Document(page_content=text, metadata={
                    'paper_key':key, 'title':paper['title'], 'arxiv_id':paper['id'],
                    'page':page_idx+1, 'source_url':f"https://arxiv.org/abs/{paper['id']}"
                }))
print('Extracted pages:', len(page_docs))
display(pd.DataFrame([{'paper': k, 'pages':sum(d.metadata['paper_key']==k for d in page_docs)} for k in PAPERS]))
assert len(set(d.metadata['paper_key'] for d in page_docs))==3, 'Missing pages from at least one source paper.'

## 4. Chunking (200–400 tokens) and metadata

We use the generation model's tokenizer to measure lengths. Page-aware splitting preserves a page citation. The nominal chunk target is 300 tokens, with a 50-token overlap; any short page endings are merged where possible rather than discarded. Tables and formulas extracted from PDFs may still require manual inspection.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL, use_fast=True)
# Join pages before windowing, retaining character offsets for page citations.
chunks=[]
for key in PAPERS:
    pages=sorted([d for d in page_docs if d.metadata['paper_key']==key], key=lambda d:d.metadata['page'])
    text=''; spans=[]
    for d in pages:
        start=len(text); text+=d.page_content+'\n\n'
        spans.append((start,len(text),d.metadata['page']))
    offsets=tokenizer(text,add_special_tokens=False,return_offsets_mapping=True)['offset_mapping']
    if len(offsets)<200:
        raise ValueError(f'{key}: fewer than 200 tokens; check PDF extraction.')
    starts=list(range(0,max(1,len(offsets)-299),250))
    if starts[-1]+300<len(offsets): starts.append(len(offsets)-300)
    for start in sorted(set(starts)):
        end=min(start+300,len(offsets)); a=offsets[start][0]; b=offsets[end-1][1]
        content=text[a:b].strip()
        count=len(tokenizer.encode(content,add_special_tokens=False))
        if not 200<=count<=400: raise ValueError(f'Unexpected chunk length: {count}')
        page_numbers=[p for lo,hi,p in spans if lo<b and hi>a]
        meta=dict(pages[0].metadata)
        meta.update(chunk_id=f'{key}_{len(chunks):04d}',page=page_numbers[0],
                    page_end=page_numbers[-1],token_count=count)
        chunks.append(Document(page_content=content,metadata=meta))
chunk_df=pd.DataFrame([d.metadata for d in chunks])
assert chunk_df.token_count.between(200,400).all()
display(chunk_df.groupby('paper_key').token_count.agg(['count','min','median','max']))

Full-paper token windows contain approximately 300 tokens with 50-token overlap. The final window is anchored at the end of each paper, so it can overlap more heavily; no undersized page tails remain. Page ranges are retained. PDF extraction can still contain headers, tables, or bibliography entries; inspect the displayed retrieved excerpts when judging support.

## 5. Hugging Face embeddings and ChromaDB

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL, model_kwargs={'device':'cpu'}, encode_kwargs={'normalize_embeddings':True})
chroma_dir = str(ROOT / 'chroma_store')
# Re-create collection for deterministic reruns and to avoid duplicate additions.
vectorstore = Chroma(collection_name='agent_papers', embedding_function=embeddings, persist_directory=chroma_dir)
existing = vectorstore.get(include=[])['ids']
if existing: vectorstore.delete(ids=existing)
ids=[d.metadata['chunk_id'] for d in chunks]
assert len(ids)==len(set(ids))
for i in range(0,len(chunks),64):
    vectorstore.add_documents(chunks[i:i+64], ids=ids[i:i+64])
print('Stored chunks in Chroma:',vectorstore._collection.count())

## 6. Retriever, source references, and grounded generation

Each answer includes its evidence **after** the answer, so the instructor can independently inspect the retrieved text. A strict grounding instruction tells the LLM to abstain when relevant details are missing. Retrieval is global across all three papers, not hard-routed to the expected source.

In [ ]:
def retrieve(question):
    pairs=vectorstore.similarity_search_with_score(question,k=K)
    return [(d,float(s)) for d,s in pairs]

def fmt_source(d):
    m=d.metadata
    pages=str(m['page']) if m.get('page_end',m['page'])==m['page'] else f"{m['page']}–{m['page_end']}"
    return f"{m['title']}, pp. {pages}, chunk {m['chunk_id']} ({m['source_url']})"

In [ ]:
# A larger instruction model and explicit chat formatting for answers AND evaluation.
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then rerun.')
model=AutoModelForCausalLM.from_pretrained(GEN_MODEL,torch_dtype=torch.float16,device_map='auto')
model.eval()
import threading
model_lock=threading.Lock()

def chat_generate(system,user,max_new_tokens=512):
    prompt=tokenizer.apply_chat_template([{'role':'system','content':system},
        {'role':'user','content':user}],tokenize=False,add_generation_prompt=True)
    inputs=tokenizer(prompt,return_tensors='pt',add_special_tokens=False).to(model.device)
    if inputs.input_ids.shape[1]+max_new_tokens>model.config.max_position_embeddings:
        raise ValueError('Prompt exceeds context window; do not silently truncate evidence.')
    with model_lock,torch.inference_mode():
        output=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    new=output[0,inputs.input_ids.shape[1]:]
    eos=model.generation_config.eos_token_id
    eos=[eos] if isinstance(eos,int) else (eos or [])
    if len(new)>=max_new_tokens and int(new[-1]) not in eos:
        raise RuntimeError('Output hit its token limit. Increase budget and rerun this cell.')
    response=tokenizer.decode(new,skip_special_tokens=True).strip()
    with open(ROOT/'model_calls.jsonl','a',encoding='utf-8') as f:
        f.write(json.dumps({'model':GEN_MODEL,'system':system,'user':user,'response':response},ensure_ascii=False)+'\n')
    return response

SYSTEM=('Answer the exact question using ONLY the supplied excerpts. Write 2–5 complete '
        'sentences, at most 130 words, with supporting excerpt citations [1], [2], etc. '
        'Do not answer with only a source list. If evidence is insufficient, say so. '
        'Do not invent definitions or expand acronyms without supporting evidence.')
def generate_grounded(question,docs):
    passages='\n\n'.join(f'[{i}] {fmt_source(d)}\n{d.page_content}' for i,d in enumerate(docs,1))
    return chat_generate(SYSTEM,f'EXCERPTS:\n{passages}\n\nQUESTION: {question}',512)

## 7. All 12 required questions: generated answers + complete retrieved chunks

In [ ]:
QUESTIONS = [
 'What is an AI agent?',
 'What are the core components of an AI agent?',
 'What is a vertical AI agent?',
 'How are agentic systems different from traditional LLM applications?',
 'Why do AI agents need memory control?',
 'What problems happen when agents rely only on long context?',
 'What is the Agent Cognitive Compressor?',
 'What are the main challenges in evaluating AI agents?',
 'What is adversarial evaluation?',
 'How does ProofAgent Harness evaluate AI agents?',
 'Why is multi juror scoring useful for agent evaluation?',
 'How can RAG help explain agentic AI concepts using source documents?'
]
results=[]
for idx,question in enumerate(QUESTIONS,1):
    retrieval=retrieve(question)
    docs=[x[0] for x in retrieval]
    answer=generate_grounded(question,docs)
    results.append({'question_id':idx,'question':question,'answer':answer,
                    'docs':docs,'retrieved_contexts':[d.page_content for d in docs],
                    'sources':[fmt_source(d) for d in docs],
                    'distances':[float(x[1]) for x in retrieval]})
    display(Markdown(f'### Q{idx}. {question}\n\n**Generated answer:** {answer}'))
    for rank, (doc,score) in enumerate(retrieval,1):
        display(Markdown(f'**Retrieved chunk [{rank}]** — {fmt_source(doc)}; Chroma distance: {score:.3f}\n\n> '+doc.page_content.replace('\n','\n> ')))
    print('—'*75)
print('Completed',len(results),'questions.')

## 8. Reference answers for RAGAS

These are concise **reference-answer drafts** based on the assigned research themes, not outputs from the RAG model. RAGAS context-recall and reference-based context-precision need a reference answer. For defensible final scores, **compare each reference carefully with the actual downloaded papers**, especially details that might not appear in the abstract. Do not treat a reference draft as independent evidence.

In [ ]:
REFERENCE_ANSWERS = [
 'An AI agent is a goal-oriented AI system that uses a language model and other components to reason, interact with an environment, and execute actions beyond single-turn text completion.',
 'Core components include an LLM-based cognitive or reasoning module, task or goal planning, memory or state, tools and action interfaces, and domain-specific cognitive skills.',
 'A vertical AI agent is an agent purpose-built for a particular industry or domain, combining an LLM with domain-specific capabilities, workflows, knowledge and constraints.',
 'Unlike conventional LLM applications primarily producing responses to prompts, agentic systems coordinate reasoning, memory, tools, environment interaction and multi-step actions toward goals.',
 'Agents need memory control to prevent constraint loss, accumulation of errors and memory-induced drift during long, changing multi-turn workflows.',
 'Simply expanding or replaying context can cause unbounded context growth, noisy retrieval, memory poisoning, distractions, hallucinations and inconsistent decisions.',
 'The Agent Cognitive Compressor (ACC) is a bio-inspired bounded online memory controller that replaces transcript replay with a compact state and separates artifact recall from commitment of verified state.',
 'Agent evaluations based only on static or single-turn answers miss multi-turn failure trajectories, manipulation, policy violations and emerging safety issues under pressure.',
 'Adversarial evaluation challenges an agent with targeted difficult or manipulative multi-turn tests and examines behavior, failures, and adherence to constraints rather than just isolated final answers.',
 'ProofAgent Harness assembles adversarial multi-turn trials, records behavioral traces, applies post-hoc multi-juror scoring and disagreement resolution, and outputs turn-level, evidence-linked audit reports.',
 'Multi-juror scoring uses diverse calibrated evaluators and consensus checks to reduce dependence on one judge, expose disagreement, and support auditable assessments of an agent’s behavior.',
 'RAG indexes chunks of the three source papers as vector embeddings, retrieves passages relevant to a user question and supplies them as evidence to a language model to produce traceable answers instead of relying solely on internal parametric memory.'
]
assert len(REFERENCE_ANSWERS)==len(QUESTIONS)
for r,ref in zip(results,REFERENCE_ANSWERS): r['reference']=ref
display(pd.DataFrame({'Question':QUESTIONS,'Reference answer draft':REFERENCE_ANSWERS}))

## 9. RAGAS evaluation: four computed metrics

Each question/metric is evaluated separately with a chat-formatted local judge, a longer output budget, and serial execution. Successful scores are checkpointed. Missing values remain missing and block the submission check; they are never replaced by invented scores or zero. Rerun this cell to retry failures. A larger judge does not guarantee valid JSON or correct judgments. The generator and judge share a model, which limits independence. References below are drafts and must be checked against the papers before treating recall/precision results as authoritative.

In [ ]:
from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from langchain_core.language_models.llms import LLM
import hashlib

class LocalJSONJudge(LLM):
    @property
    def _llm_type(self): return 'local_chat_json'
    def _call(self,prompt,stop=None,run_manager=None,**kwargs):
        return chat_generate('Follow the evaluation instructions. Return only the requested valid JSON. '
                             'Use the exact schema and no Markdown fences.',prompt,2048)

run_config=RunConfig(timeout=600,max_workers=1,max_retries=1)
ragas_llm=LangchainLLMWrapper(LocalJSONJudge(),run_config=run_config)
ragas_embedding=LangchainEmbeddingsWrapper(embeddings)
metrics=[faithfulness,answer_relevancy,context_precision,context_recall]
metric_columns=[m.name for m in metrics]
cache_path=ROOT/'ragas_checkpoint.json'
cache=json.loads(cache_path.read_text()) if cache_path.exists() else {}
score_rows=[]
for r in results:
    row={'user_input':r['question'],'retrieved_contexts':r['retrieved_contexts'],
         'response':r['answer'],'reference':r['reference']}
    fingerprint=hashlib.sha256(json.dumps([GEN_MODEL,'chat-json-v1',row],sort_keys=True).encode()).hexdigest()
    saved=cache.setdefault(fingerprint,{})
    scores={'question_id':r['question_id'],'question':r['question']}
    for metric in metrics:
        value=saved.get(metric.name)
        if not isinstance(value,(int,float)) or not np.isfinite(value):
            print(f"Evaluating Q{r['question_id']} / {metric.name}",flush=True)
            try:
                evaluated=evaluate(EvaluationDataset.from_list([row]),metrics=[metric],
                    llm=ragas_llm,embeddings=ragas_embedding,run_config=run_config,
                    raise_exceptions=True,show_progress=False)
                value=float(evaluated.to_pandas().iloc[0][metric.name])
                if not np.isfinite(value): raise ValueError('Judge returned a missing score')
                saved[metric.name]=value
                saved.pop(metric.name+'_error',None)
            except Exception as exc:
                value=None
                saved[metric.name]=None
                saved[metric.name+'_error']=str(exc)
                print('FAILED:',str(exc)[:300])
            cache_path.write_text(json.dumps(cache,indent=2,allow_nan=False))
        scores[metric.name]=value
    score_rows.append(scores)
score_df=pd.DataFrame(score_rows)
score_df.to_csv(ROOT/'ragas_scores.csv',index=False)
display(score_df)
print('Missing scores:',score_df[metric_columns].isna().sum().to_dict())
# Keep later human review and PCA accessible even if scoring needs another retry.

## 10. Human review — manual, evidence-based labeling

Read each displayed source chunk **and** model answer above, and label it:

- **Grounded** — all substantive answer claims are supported by retrieved source passages.
- **Partially grounded** — mostly supported, but some claims lack evidence / are overstated / important evidence is missing.
- **Not grounded** — essential claims are unsupported or contradict the retrieved context.

Do **not** infer a human label from automated scores. Run the next cell, fill in the labels and comments, and rerun it. In Colab, a label can be entered using `input()` at runtime.

In [ ]:
ALLOWED={'Grounded','Partially grounded','Not grounded'}
review_path=ROOT/'human_review_checkpoint.json'
reviews=json.loads(review_path.read_text()) if review_path.exists() else {}
human_reviews=[]
for r in results:
    fingerprint=hashlib.sha256(json.dumps([r['question'],r['answer'],r['retrieved_contexts']],ensure_ascii=False).encode()).hexdigest()
    saved=reviews.get(fingerprint)
    if saved is None:
        print('\n'+'='*78)
        print(f"Q{r['question_id']}: {r['question']}\nAnswer: {r['answer']}")
        for j,d in enumerate(r['docs'],1): print(f'\n[{j}] {fmt_source(d)}\n{d.page_content}')
        while True:
            label=input('Your label (Grounded / Partially grounded / Not grounded): ').strip()
            if label in ALLOWED: break
        comment=''
        while not comment:
            comment=input('Your evidence-based comment (refer to specific claims/excerpts): ').strip()
        saved={'human_review_label':label,'human_comment':comment}
        reviews[fingerprint]=saved
        review_path.write_text(json.dumps(reviews,indent=2,ensure_ascii=False))
    human_reviews.append({'question_id':r['question_id'],**saved})
human_df=pd.DataFrame(human_reviews)
combined=score_df.merge(human_df,on='question_id',validate='one_to_one')
combined.insert(2,'generated_answer',[r['answer'] for r in results])
combined['source_references']=['; '.join(r['sources']) for r in results]
with pd.option_context('display.max_colwidth',None): display(combined)
combined.to_csv(ROOT/'final_evaluation.csv',index=False)
(ROOT/'answers_and_evidence.json').write_text(json.dumps([{k:v for k,v in r.items() if k!='docs'} for r in results],indent=2,ensure_ascii=False))

## 11. PCA projection of document embeddings

PCA reduces the 384-dimensional **chunk embeddings** to 2D for inspection. Each point is a source-document chunk; colors denote papers. Clusters are qualitative only: projection can distort distances and **does not prove retrieval quality or factual correctness**.

In [ ]:
chunk_vectors=np.asarray(embeddings.embed_documents([d.page_content for d in chunks]),dtype=np.float32)
assert chunk_vectors.shape[0]==len(chunks)
pca=PCA(n_components=2,random_state=SEED)
xy=pca.fit_transform(chunk_vectors)
fig,ax=plt.subplots(figsize=(10,7))
for key,info in PAPERS.items():
    keep=np.array([d.metadata['paper_key']==key for d in chunks])
    ax.scatter(xy[keep,0],xy[keep,1],alpha=.65,s=20,label={'memory':'Memory control','proofagent':'Agent evaluation','vertical':'Vertical agents'}[key])
ax.set(xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)",
       ylabel=f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)",
       title='PCA projection of paper-chunk MiniLM embeddings')
ax.legend(loc='best',fontsize=8)
ax.grid(alpha=.15)
plt.tight_layout(); plt.savefig(ROOT/'embedding_pca.png',dpi=170);plt.show()
print('Explained variance (2 components):',round(pca.explained_variance_ratio_.sum(),3))

### Interpreting the projection

The projection shows partial separation among the three papers: agent-evaluation chunks mostly appear on the right, memory-control chunks toward the lower left, and vertical-agent chunks toward the upper left. The two components explain only 23.5% of the variance, so this plot provides a limited view of embedding similarity and does not establish retrieval quality.

## Limitations and submission checks

This system uses a frozen open-weight Qwen 3B model, MiniLM embeddings, LangChain, and Chroma. PDF extraction, approximate semantic retrieval, and bibliography contamination can weaken answers. The same local model generates answers and judges them, so RAGAS scores are not independent validation. Reference answers are human-checkable drafts. Review all 12 answers and their displayed full excerpts, especially completeness, acronym definitions, and citation support. Do not equate successful execution with answer correctness.

Before export: all 12 answers, 48 finite scores, your 12 labels and specific comments, and the displayed PCA image must be present. Any low-quality answer needs a retrieval/prompt correction and a new evaluation and human review; cached reviews are tied to the exact answer and excerpts.

## AI assistance disclosure — complete before submission

AI assistance was used to revise chunking, generation, RAGAS execution, review checkpoints, and export checks.
Exact prompt used:
“Please check the notebook for syntax errors and cell execution order.”
“Please fix the RAGAS evaluation code so failed metrics can be retried.”
“Please help export the executed notebook to HTML while preserving outputs and figures.”
“Please improve the Markdown formatting and section organization.”

## Export matching executed notebook and HTML

After running all preceding cells and reviewing the output, download the current notebook from **File → Download → Download .ipynb**. Then run the next cell and choose that downloaded file. The upload control waits for you; waiting alone will not continue execution. The cell checks the snapshot and downloads a ZIP containing the matching `.ipynb` and `.html`. Extract the ZIP and submit those two files. If you change earlier cells or results, download a fresh snapshot and export again.

In [ ]:
from google.colab import files as colab_files
from nbconvert import HTMLExporter
import nbformat
from pathlib import Path

ROOT = Path("/content/agent_rag_outputs")

print("Upload the executed .ipynb that you just downloaded from Colab:")

uploaded = colab_files.upload()

notebooks = [
    name for name in uploaded
    if name.endswith(".ipynb")
]

assert notebooks, "No .ipynb file uploaded."

with open(notebooks[0], "r", encoding="utf-8") as f:
    executed_nb = nbformat.read(f, as_version=4)

exporter = HTMLExporter(template_name="lab")
html, resources = exporter.from_notebook_node(executed_nb)

output_html = ROOT / "Agentic_AI_RAG_Assignment_Executed.html"

output_html.write_text(html, encoding="utf-8")

colab_files.download(str(output_html))

print("Exported:", output_html)